In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn import preprocessing
from dgl.data import DGLDataset
import dgl
import time
import networkx as nx
import category_encoders as ce
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import torch
import tqdm
import math

from typing import *
from sklearn.preprocessing import StandardScaler, Normalizer
import socket
import struct
import random
from sklearn.model_selection import train_test_split

## Loading Graphs
* Multigraph with
    - Edge features
        - h : A list of features in data
        - Label (0-1)
        - Attack
    - Node features
        - {1,.., 1} same length as h

In [2]:
file_path = "test.graph"
# Use dgl.save_graphs to save the graph to the specified file
##dgl.save_graphs(file_path, [test_g])
# Use dgl.load_graphs to load the graph from the file
test_g, _ = dgl.load_graphs(file_path)
# The loaded_graphs variable now contains the loaded DGLGraph(s)
test_g = test_g[0]  # Assuming you saved a single graph

# Specify the path to the saved graph file
file_path = "train.graph"
# Use dgl.save_graphs to save the graph to the specified file
##dgl.save_graphs(file_path, [train_g])
# Use dgl.load_graphs to load the graph from the file
train_g, _ = dgl.load_graphs(file_path)
# The loaded_graphs variable now contains the loaded DGLGraph(s)
train_g = train_g[0]  # Assuming you saved a single graph

In [3]:
import joblib
lab_enc = joblib.load('gnn_label_encoder.pkl')

C:\Users\pc\anaconda3\envs\xcba\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


# Self-Supervised Learning
### E-GraphSAGE

In [4]:
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import tqdm
import gc

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
      super(SAGELayer, self).__init__()
      self.W_apply = nn.Linear(ndim_in + edims , ndim_out)
      self.activation = F.relu
      self.W_edge = nn.Linear(128 * 2, 256)
      self.reset_parameters()

    def reset_parameters(self):
      """
      Reset parameters whenever object created
      """
      gain = nn.init.calculate_gain('relu')
      nn.init.xavier_uniform_(self.W_apply.weight, gain=gain)

    def message_func(self, edges):
      """
      It sends the 'h' feature data from edges to nodes
      """
      return {'m':  edges.data['h']}

    def forward(self, g_dgl, nfeats, efeats):
      """
      update_all : message aggregation
      applies a linear transformation
      concatenates node features with aggregated neighbor features
      then applies a non-linear activation function (ReLU)
      """
      with g_dgl.local_scope():
        g = g_dgl
        g.ndata['h'] = nfeats
        g.edata['h'] = efeats
        g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
        g.ndata['h'] = F.relu(self.W_apply(torch.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))

        # Compute edge embeddings
        u, v = g.edges()
        edge = self.W_edge(torch.cat((g.srcdata['h'][u], g.dstdata['h'][v]), 2))
        return g.ndata['h'], edge

In [5]:
class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim,  activation):
      super(SAGE, self).__init__()
      self.layers = nn.ModuleList()
      self.layers.append(SAGELayer(ndim_in, edim, 128, F.relu))

    def forward(self, g, nfeats, efeats, corrupt=False):
      """
      If corruption : permutate edge features
      Then send data into layers to find node&edge features
      """
      if corrupt:
        e_perm = torch.randperm(g.number_of_edges())
        efeats = efeats[e_perm]
      for i, layer in enumerate(self.layers):
        nfeats, e_feats = layer(g, nfeats, efeats)
      return nfeats.sum(1), e_feats.sum(1)

# Self-Supervised Learning
### Deep Graph Infomax (DGI)

In [6]:
class Discriminator(nn.Module):
    def __init__(self, n_hidden):
        super(Discriminator, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(n_hidden, n_hidden))
        self.reset_parameters()

    def uniform(self, size, tensor):
        bound = 1.0 / math.sqrt(size)
        if tensor is not None:
            tensor.data.uniform_(-bound, bound)

    def reset_parameters(self):
        size = self.weight.size(0)
        self.uniform(size, self.weight)

    def forward(self, features, summary):
        features = torch.matmul(features, torch.matmul(self.weight, summary))
        return features

In [7]:
class DGI(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
        super(DGI, self).__init__()
        self.encoder = SAGE(ndim_in, ndim_out, edim,  F.relu)
        #self.discriminator = Discriminator(128)
        self.discriminator = Discriminator(256)
        self.loss = nn.BCEWithLogitsLoss()

    # def forward(self, graph, feat, e_features, edge_weight=None, embed=False):
    #     feat = torch.reshape(train_g.ndata['h'],
    #                                (train_g.ndata['h'].shape[0], 1,
    #                                 39))
    #     positive = self.encoder(graph, feat, e_features, corrupt=False)
    #     negative = self.encoder(graph, feat, e_features, corrupt=True)
    #     self.loss = nn.BCEWithLogitsLoss()

    def forward(self, graph, feat, e_features, edge_weight=None, eweight=None, embed=False):
 
        feat = torch.reshape(train_g.ndata['h'],
                                   (train_g.ndata['h'].shape[0], 1,
                                    39))
        positive = self.encoder(graph, feat, e_features, corrupt=False)
        negative = self.encoder(graph, feat, e_features, corrupt=True)
        if embed:
            return e_features
            #return feat

        positive = positive[1]
        negative = negative[1]

        summary = torch.sigmoid(positive.mean(dim=0))

        positive = self.discriminator(positive, summary)
        negative = self.discriminator(negative, summary)

        l1 = self.loss(positive, torch.ones_like(positive))
        l2 = self.loss(negative, torch.zeros_like(negative))

        return torch.tensor([[l1+l2, 0]],requires_grad=True)

## Training DGI
* Same hyperparameters and optimizer specified in the "Anomal-E".

In [8]:
ndim_in = train_g.ndata['h'].shape[1]
hidden_features = 128
ndim_out = 128
num_layers = 1
edim = train_g.edata['h'].shape[1]
learning_rate = 1e-3
epochs = 4000

In [9]:
dgi = DGI(ndim_in,
    ndim_out,
    edim,
    F.relu)

dgi_optimizer = torch.optim.Adam(dgi.parameters(),
                lr=1e-3,
                weight_decay=0.)

In [10]:
# Format node and edge features for E-GraphSAGE
train_g.ndata['h'] = torch.reshape(train_g.ndata['h'],
                                   (train_g.ndata['h'].shape[0], 1,
                                    train_g.ndata['h'].shape[1]))

train_g.edata['h'] = torch.reshape(train_g.edata['h'],
                                   (train_g.edata['h'].shape[0], 1,
                                    train_g.edata['h'].shape[1]))

In [11]:
# Convert to GPU
train_g = train_g

In [12]:
node_features = train_g.ndata['h']
edge_features = train_g.edata['h']


## Loading trained DGI
* Same hyperparameters and optimizer specified in the "Anomal-E".

In [13]:
dgi.load_state_dict(torch.load('best_dgi.pkl'))

C:\Users\pc\AppData\Local\Temp\ipykernel_19448\597080661.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dgi.load_state_dict(torch.load('best_dgi.pkl'))


<All keys matched successfully>

In [14]:
dgi(train_g, node_features, edge_features)

tensor([[0.0757, 0.0000]], requires_grad=True)

In [15]:
train_g

Graph(num_nodes=28070, num_edges=699060,
      ndata_schemes={'h': Scheme(shape=(1, 39), dtype=torch.float32)}
      edata_schemes={'Label': Scheme(shape=(), dtype=torch.int64), 'Attack': Scheme(shape=(), dtype=torch.int64), 'h': Scheme(shape=(1, 39), dtype=torch.float32)})

## Edge Embeddings


* Training DGI 
* Seperate encoders are trained for training and testing graph
* After encoding train and test graphs, encodings are converted into dataframe

In [16]:
training_emb = dgi.encoder(train_g, train_g.ndata['h'], train_g.edata['h'])[1]
training_emb = training_emb.detach().cpu().numpy()

In [17]:
test_g.ndata['h'] = torch.reshape(test_g.ndata['h'],
                                   (test_g.ndata['h'].shape[0], 1,
                                    test_g.ndata['h'].shape[1]))



test_g.edata['h'] = torch.reshape(test_g.edata['h'],
                                   (test_g.edata['h'].shape[0], 1,
                                    test_g.edata['h'].shape[1]))

In [18]:
# Convert to GPU
test_g = test_g

In [19]:
testing_emb = dgi.encoder(test_g, test_g.ndata['h'], test_g.edata['h'])[1]
testing_emb = testing_emb.detach().cpu().numpy()

In [20]:
df_train = pd.DataFrame(training_emb, )
df_train["Attack"] = lab_enc.inverse_transform(
        train_g.edata['Attack'].detach().cpu().numpy())
df_train["Label"] = train_g.edata['Label'].detach().cpu().numpy()

df_test = pd.DataFrame(testing_emb, )
df_test["Attack"] = lab_enc.inverse_transform(
        test_g.edata['Attack'].detach().cpu().numpy())
df_test["Label"] = test_g.edata['Label'].detach().cpu().numpy()

In [21]:
df_train # Edge features, labels

,0,1,2,3,4,5,6,7,8,9,...,248,249,250,251,252,253,254,255,Attack,Label
0,0.002874,0.020945,-0.080155,0.017683,-0.027178,-0.034828,0.100208,0.066146,0.015666,0.016851,...,0.015789,0.022639,-0.021940,0.043273,0.035339,-0.021965,-0.039102,0.052328,Benign,0
1,-0.011380,0.000875,-0.060205,0.025181,-0.025974,-0.043176,0.091904,0.050979,0.012244,0.001137,...,0.020061,0.041199,-0.035434,0.032933,0.011661,0.001811,-0.027890,0.039528,Benign,0
2,-0.011380,0.000875,-0.060205,0.025181,-0.025974,-0.043176,0.091904,0.050979,0.012244,0.001137,...,0.020061,0.041199,-0.035434,0.032933,0.011661,0.001811,-0.027890,0.039528,Benign,0
3,-0.011380,0.000875,-0.060205,0.025181,-0.025974,-0.043176,0.091904,0.050979,0.012244,0.001137,...,0.020061,0.041199,-0.035434,0.032933,0.011661,0.001811,-0.027890,0.039528,Benign,0
4,-0.011380,0.000875,-0.060205,0.025181,-0.025974,-0.043176,0.091904,0.050979,0.012244,0.001137,...,0.020061,0.041199,-0.035434,0.032933,0.011661,0.001811,-0.027890,0.039528,Benign,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699055,-0.012260,0.001837,-0.061892,0.025375,-0.021196,-0.037549,0.092967,0.054119,0.013441,-0.000411,...,0.019225,0.041841,-0.037635,0.026823,0.010458,0.000331,-0.034085,0.040779,Benign,0
699056,-0.012702,-0.006457,-0.035627,0.000794,-0.031555,-0.041894,0.088394,0.020692,0.024180,0.000256,...,0.019810,0.021395,-0.038149,0.019373,0.031309,0.000774,-0.023803,0.019051,Benign,0
699057,-0.011990,0.001800,-0.061707,0.025481,-0.021083,-0.037971,0.092847,0.053641,0.013735,-0.000776,...,0.019045,0.042255,-0.038323,0.026215,0.010114,0.000181,-0.033827,0.040271,Benign,0
699058,-0.020986,-0.006903,-0.040520,0.064232,-0.039967,-0.064006,0.111022,0.084072,0.006957,0.001660,...,0.031092,0.038724,-0.029109,0.047181,-0.015772,0.003680,-0.009463,0.045146,Benign,0


In [22]:
df_test

,0,1,2,3,4,5,6,7,8,9,...,248,249,250,251,252,253,254,255,Attack,Label
0,-0.014182,0.001872,-0.061665,0.024839,-0.024640,-0.043166,0.093257,0.052298,0.010392,-0.003912,...,0.023274,0.045358,-0.036305,0.029138,0.005451,0.007660,-0.032484,0.037503,Benign,0
1,-0.014182,0.001872,-0.061665,0.024839,-0.024640,-0.043166,0.093257,0.052298,0.010392,-0.003912,...,0.023274,0.045358,-0.036305,0.029138,0.005451,0.007660,-0.032484,0.037503,Benign,0
2,-0.013706,0.001825,-0.061401,0.024746,-0.024921,-0.043408,0.093105,0.051739,0.010864,-0.003393,...,0.022803,0.045019,-0.036684,0.028938,0.005754,0.006736,-0.032026,0.037430,Benign,0
3,-0.013706,0.001825,-0.061401,0.024746,-0.024921,-0.043408,0.093105,0.051739,0.010864,-0.003393,...,0.022803,0.045019,-0.036684,0.028938,0.005754,0.006736,-0.032026,0.037430,Benign,0
4,-0.013706,0.001825,-0.061401,0.024746,-0.024921,-0.043408,0.093105,0.051739,0.010864,-0.003393,...,0.022803,0.045019,-0.036684,0.028938,0.005754,0.006736,-0.032026,0.037430,Benign,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299573,-0.016143,0.002402,-0.066723,0.033957,-0.023753,-0.040508,0.094806,0.063267,0.005500,0.000264,...,0.020979,0.039151,-0.030941,0.034273,0.002954,0.001018,-0.030535,0.045807,Benign,0
299574,-0.015474,-0.014349,-0.039710,0.062475,-0.041624,-0.074456,0.109776,0.077543,0.008884,0.007660,...,0.021809,0.034798,-0.028810,0.044246,-0.009549,0.000432,-0.001217,0.048261,Benign,0
299575,-0.016513,-0.012060,-0.035600,0.061889,-0.041761,-0.071070,0.112251,0.078589,0.011440,0.005799,...,0.025984,0.037767,-0.031845,0.043916,-0.012022,0.000221,-0.004957,0.046747,Benign,0
299576,-0.012318,0.002306,-0.059723,0.033823,-0.024784,-0.041274,0.099030,0.063232,0.011299,0.001084,...,0.021974,0.041263,-0.035929,0.030141,0.005817,-0.002031,-0.029932,0.044067,Benign,0


## GNNExplainer


In [23]:
df_gxai = pd.DataFrame(columns = ["GNNExplainer","PGExplainer"])

In [24]:
node_features

tensor([[[1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.]],

        ...,

        [[1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.]]])

In [25]:
import dgl.function as fn
import torch
import torch.nn as nn
from dgl.data import GINDataset
from dgl.dataloading import GraphDataLoader
from dgl.nn import AvgPooling, GNNExplainer, PGExplainer, SubgraphX

In [26]:
# Explain the prediction for graph 0
explainer = GNNExplainer(dgi, num_hops=2)

In [27]:
g = train_g
features = g.ndata['h']
edge_features = g.edata['h']
gnn_kwargs = {
    'e_features': edge_features,  
}

f = torch.reshape(features,(train_g.ndata['h'].shape[0], 39))


feat_mask, edge_mask = explainer.explain_graph(g, f, **gnn_kwargs)

Explain graph: 100%|█████████████████████| 100/100 [06:24<00:00,  3.85s/it]


In [29]:
train_g

Graph(num_nodes=28070, num_edges=699060,
      ndata_schemes={'h': Scheme(shape=(1, 39), dtype=torch.float32)}
      edata_schemes={'Label': Scheme(shape=(), dtype=torch.int64), 'Attack': Scheme(shape=(), dtype=torch.int64), 'h': Scheme(shape=(1, 39), dtype=torch.float32)})

In [30]:
feat_mask

tensor([0.2530, 0.2606, 0.2749, 0.2461, 0.2600, 0.2370, 0.2549, 0.2784, 0.3067,
        0.2772, 0.2379, 0.3236, 0.2860, 0.3033, 0.2711, 0.2807, 0.2760, 0.2831,
        0.2672, 0.2796, 0.3256, 0.2547, 0.2411, 0.2478, 0.2780, 0.2638, 0.2622,
        0.2896, 0.2880, 0.2725, 0.2644, 0.3129, 0.2672, 0.2506, 0.2824, 0.2384,
        0.2754, 0.2558, 0.2824])

In [31]:
edge_mask

tensor([0.2753, 0.2758, 0.2752,  ..., 0.2757, 0.2746, 0.2768])

In [32]:
len(feat_mask)

39

In [33]:
len(edge_mask)

699060

In [34]:
edge_mask.shape

torch.Size([699060])

In [35]:
df_gxai['GNNExplainer'] = edge_mask.numpy()

In [36]:
df_gxai.head(5)

,GNNExplainer,PGExplainer
0,0.275305,NaN
1,0.275772,NaN
2,0.275177,NaN
3,0.273810,NaN
4,0.273337,NaN


In [37]:
dgi(g, features, edge_features)[0].log_softmax(dim=-1)

tensor([-0.6562, -0.7315], grad_fn=<LogSoftmaxBackward0>)

In [38]:
train_g.ndata['h'].shape

torch.Size([28070, 1, 39])

In [39]:
torch.topk(edge_mask.flatten(), 5).indices

tensor([573237, 110195, 223399, 557990, 507460])

## PGExplainer


In [40]:
g = train_g
features = g.ndata['h']
edge_features = g.edata['h']
gnn_kwargs = {
    'e_features': edge_features,  
}

f = torch.reshape(features,(train_g.ndata['h'].shape[0], 39))


In [41]:
dgi.encoder

SAGE(
  (layers): ModuleList(
    (0): SAGELayer(
      (W_apply): Linear(in_features=78, out_features=128, bias=True)
      (W_edge): Linear(in_features=256, out_features=256, bias=True)
    )
  )
)

In [42]:
explainer = PGExplainer(dgi, num_features=39, num_hops=2)

probs, edge_weight = explainer.explain_graph(g, f, **gnn_kwargs, training=True)

In [43]:
edge_weight

tensor([0.3536, 0.4998, 0.1364,  ..., 0.4889, 0.1823, 0.4867],
       grad_fn=<DivBackward0>)

In [44]:
len(set(edge_weight.detach().numpy()))

580986

In [45]:
df_gxai['PGExplainer'] = edge_weight.detach().numpy()

In [46]:
df_gxai

,GNNExplainer,PGExplainer
0,0.275305,0.353601
1,0.275772,0.499767
2,0.275177,0.136441
3,0.273810,0.064968
4,0.273337,0.437619
...,...,...
699055,0.278417,0.649225
699056,0.271927,0.244231
699057,0.275749,0.488901
699058,0.274561,0.182299


In [47]:
df_gxai.to_parquet("graph_explainers_df.parquet")

## SubgraphX

In [48]:
# explainer = SubgraphX(dgi, num_hops=1)

In [49]:
# ## Subgraph
# #graph = dgl.node_subgraph(train_g, random.sample(range(0, train_g.num_nodes()+1), 100))

# g = train_g
# features = g.ndata['h']
# edge_features = g.edata['h']
# l = torch.tensor(0)

# gnn_kwargs = {
#     'e_features': edge_features,
# }

# f = torch.reshape(features,(g.ndata['h'].shape[0], 39))

In [50]:
# g_nodes_explain = explainer.explain_graph(g, f, target_class=l, **gnn_kwargs)